In [1]:
import os
import warnings

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""
warnings.filterwarnings("ignore")

# === 1. Patch httpx.Client at class level ===
# Affects ALL clients regardless of when they were created — covers
# huggingface_hub's internally cached session too.
import httpx

if not getattr(httpx.Client.__init__, '_ssl_patched', False):
    _orig_client_init = httpx.Client.__init__
    def _patched_client_init(self, *args, **kwargs):
        kwargs['verify'] = False
        _orig_client_init(self, *args, **kwargs)
    _patched_client_init._ssl_patched = True
    httpx.Client.__init__ = _patched_client_init

if not getattr(httpx.AsyncClient.__init__, '_ssl_patched', False):
    _orig_async_init = httpx.AsyncClient.__init__
    def _patched_async_init(self, *args, **kwargs):
        kwargs['verify'] = False
        _orig_async_init(self, *args, **kwargs)
    _patched_async_init._ssl_patched = True
    httpx.AsyncClient.__init__ = _patched_async_init

# === 2. Force huggingface_hub to rebuild its cached session ===
# configure_http_backend tells HF to use our factory for new sessions.
# Clearing the cached _SESSION variable forces it to create a fresh one
# via our factory on the next call, rather than reusing the old one.
try:
    from huggingface_hub import configure_http_backend
    configure_http_backend(backend_factory=lambda: httpx.Client(verify=False))
    print("✅ HF backend factory set")
except Exception as e:
    print(f"⚠️  configure_http_backend unavailable: {e}")

try:
    import huggingface_hub.utils._http as _hf_http
    for attr in ['_SESSION', '_session', 'SESSION']:
        if hasattr(_hf_http, attr):
            setattr(_hf_http, attr, None)
            print(f"✅ Cleared cached HF session ({attr})")
except Exception as e:
    print(f"⚠️  Could not clear HF session cache: {e}")

# === 3. requests fallback (some older deps still use it) ===
import urllib3, requests
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
if not getattr(requests.Session.request, '_ssl_patched', False):
    _orig_req = requests.Session.request
    def _patched_req(self, method, url, **kwargs):
        kwargs['verify'] = False
        return _orig_req(self, method, url, **kwargs)
    _patched_req._ssl_patched = True
    requests.Session.request = _patched_req

import huggingface_hub
print(f"huggingface_hub version: {huggingface_hub.__version__}")
print("✅ SSL bypass fully activated.")

⚠️  configure_http_backend unavailable: cannot import name 'configure_http_backend' from 'huggingface_hub' (/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/huggingface_hub/__init__.py)
huggingface_hub version: 1.20.1
✅ SSL bypass fully activated.


In [2]:
import re, json, sys
import pandas as pd
from tqdm import tqdm
import gc

from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler
import mlx.nn as nn

# ── Patch 1: non-strict weight loading (Gemma 4 KV-sharing layers)
# Idempotent — safe to re-run Cell 2 without kernel restart
if not getattr(nn.Module.load_weights, '_patched_non_strict', False):
    _original_load_weights = nn.Module.load_weights
    def _non_strict_load_weights(self, file_or_weights, strict=True):
        return _original_load_weights(self, file_or_weights, strict=False)
    _non_strict_load_weights._patched_non_strict = True
    nn.Module.load_weights = _non_strict_load_weights

# ── Patch 2: gemma4_unified shim (12B / 26B / 27B use this model_type)
import mlx_lm.models.gemma4 as _gemma4_module
sys.modules['mlx_lm.models.gemma4_unified'] = _gemma4_module

# ── Paths
INPUT_DIR     = '/Users/nejam_abderrahmane/Downloads/Split_Files'
EXCEL_PATH    = os.path.join(INPUT_DIR, 'Mercari.xlsx')
TAXONOMY_PATH = '/Users/nejam_abderrahmane/Desktop/Code Analysis/my_taxonomy.json'
OUTPUT_DIR    = '/Users/nejam_abderrahmane/Downloads/Gemma4_Benchmarks'
os.makedirs(OUTPUT_DIR, exist_ok=True)
TITLE_COLUMN  = "item_name"

# Greedy sampler — built once, shared across all model cells
greedy_sampler = make_sampler(temp=0.0)

# ── Load taxonomy
print("Loading taxonomy...")
try:
    with open(TAXONOMY_PATH, "r", encoding="utf-8") as f:
        taxonomy = json.load(f)
    print("✅ Taxonomy loaded.")
except FileNotFoundError:
    print("❌ Taxonomy file not found.")
    taxonomy = {}

# ── Load benchmark data
print("Loading data...")
try:
    df = pd.read_excel(EXCEL_PATH)
    test_df = df.head(100).copy().reset_index(drop=True)
    print(f"✅ {len(test_df)} items ready.")
except Exception as e:
    print(f"❌ Error loading Excel: {e}")
    test_df = None

# ── Strip Gemma 4 thinking blocks from raw output
def strip_thinking(text: str) -> str:
    text = re.sub(r"<\|channel>thought.*?<channel\|>", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|channel>\w*", "", text)
    text = text.replace("<channel|>", "")
    return text.strip()

# ── Load a model
def load_gemma4(model_id: str):
    print(f"  Loading {model_id}...")
    model, tokenizer = load(model_id)
    return model, tokenizer

# ── Ask the model to pick one category from a list
def ask_gemma_category(product, options, level_name, model, tokenizer, sampler, parent_context=""):
    options_str = "\n".join([f"- {opt}" for opt in options])
    context_str = f"Parent Category Path: {parent_context}\n" if parent_context else ""
    system = (
        "You are a strict classification API.\n"
        "Output ONLY the exact category name from the provided list.\n"
        "NO explanations, NO extra text, NO punctuation outside the category name."
    )
    user = (
        f"{context_str}Valid {level_name} Categories:\n{options_str}\n\n"
        f"Task: Classify this product into exactly one category above.\n"
        f'Product: "{product}"\nCategory:'
    )
    messages = [{"role": "user", "content": f"{system}\n\n{user}"}]

    try:
        try:
            prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, enable_thinking=False
            )
            response = generate(model, tokenizer, prompt=prompt,
                                max_tokens=30, sampler=sampler, verbose=False)
        except TypeError:
            prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
            response = generate(model, tokenizer, prompt=prompt,
                                max_tokens=300, sampler=sampler, verbose=False)
            response = strip_thinking(response)

        prediction = response.strip().replace('"', '').replace("'", "").lstrip('- ')
        prediction = next((l.strip() for l in prediction.splitlines() if l.strip()), "")
        return prediction

    except Exception as e:
        print(f"  Generation error: {e}")
        return "Error"

# ── Walk the taxonomy tree level by level
def classify_product(product, taxonomy_tree, model, tokenizer, sampler):
    cat1, cat2, cat3 = "Other", "Other", "Other"

    def best_match(prediction, valid_options):
        if not prediction or prediction == "Error":
            return None
        pred = prediction.strip().lower()
        for opt in valid_options:
            if opt.strip().lower() == pred: return opt
        for opt in valid_options:
            if opt.strip().lower() in pred: return opt
        return None

    l1_opts    = list(taxonomy_tree.keys())
    cat1_match = best_match(ask_gemma_category(product, l1_opts, "Level 1", model, tokenizer, sampler), l1_opts)
    if not cat1_match: return cat1, cat2, cat3
    cat1 = cat1_match

    l2_data    = taxonomy_tree[cat1]
    l2_opts    = list(l2_data.keys()) if isinstance(l2_data, dict) else l2_data
    cat2_match = best_match(ask_gemma_category(product, l2_opts, "Level 2", model, tokenizer, sampler, cat1), l2_opts)
    if not cat2_match or not isinstance(l2_data, dict): return cat1, cat2, cat3
    cat2 = cat2_match

    l3_data = l2_data[cat2]
    if isinstance(l3_data, (dict, list)):
        l3_opts    = list(l3_data.keys()) if isinstance(l3_data, dict) else l3_data
        cat3_match = best_match(ask_gemma_category(product, l3_opts, "Level 3", model, tokenizer, sampler, f"{cat1} > {cat2}"), l3_opts)
        if cat3_match: cat3 = cat3_match

    return cat1, cat2, cat3

print("✅ All helpers ready.")

Loading taxonomy...
✅ Taxonomy loaded.
Loading data...
✅ 100 items ready.
✅ All helpers ready.


In [ ]:
MODEL_ID = "mlx-community/gemma-4-e2b-it-bf16"
OUT_FILE  = os.path.join(OUTPUT_DIR, "Gemma4_E2B_Predictions.xlsx")

model, tokenizer = load_gemma4(MODEL_ID)
sanity_raw = generate(model, tokenizer,
    prompt=tokenizer.apply_chat_template([{"role": "user", "content": "Who are you?"}], add_generation_prompt=True),
    max_tokens=200, sampler=greedy_sampler, verbose=False)
print(f"Sanity stripped: {repr(strip_thinking(sanity_raw))}")

results = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="E2B"):
    product_name = str(row.get(TITLE_COLUMN, "Unknown"))
    c1, c2, c3 = classify_product(product_name, taxonomy, model, tokenizer, greedy_sampler)
    results.append({"item_name": product_name, "category_1_pred": c1, "category_2_pred": c2, "category_3_pred": c3})

pd.DataFrame(results).to_excel(OUT_FILE, index=False)
print(f"✅ E2B done → {OUT_FILE}")
del model, tokenizer; gc.collect()

In [ ]:
MODEL_ID = "mlx-community/gemma-4-e4b-it-bf16"
OUT_FILE  = os.path.join(OUTPUT_DIR, "Gemma4_E4B_Predictions.xlsx")

model, tokenizer = load_gemma4(MODEL_ID)
sanity_raw = generate(model, tokenizer,
    prompt=tokenizer.apply_chat_template([{"role": "user", "content": "Who are you?"}], add_generation_prompt=True),
    max_tokens=200, sampler=greedy_sampler, verbose=False)
print(f"Sanity stripped: {repr(strip_thinking(sanity_raw))}")

results = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="E4B"):
    product_name = str(row.get(TITLE_COLUMN, "Unknown"))
    c1, c2, c3 = classify_product(product_name, taxonomy, model, tokenizer, greedy_sampler)
    results.append({"item_name": product_name, "category_1_pred": c1, "category_2_pred": c2, "category_3_pred": c3})

pd.DataFrame(results).to_excel(OUT_FILE, index=False)
print(f"✅ E4B done → {OUT_FILE}")
del model, tokenizer; gc.collect()

In [ ]:

MODEL_ID = "mlx-community/gemma-4-12b-it-4bit"
OUT_FILE  = os.path.join(OUTPUT_DIR, "Gemma4_12B_Predictions.xlsx")

model, tokenizer = load_gemma4(MODEL_ID)
sanity_raw = generate(model, tokenizer,
    prompt=tokenizer.apply_chat_template([{"role": "user", "content": "Who are you?"}], add_generation_prompt=True),
    max_tokens=200, sampler=greedy_sampler, verbose=False)
print(f"Sanity stripped: {repr(strip_thinking(sanity_raw))}")

results = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="12B"):
    product_name = str(row.get(TITLE_COLUMN, "Unknown"))
    c1, c2, c3 = classify_product(product_name, taxonomy, model, tokenizer, greedy_sampler)
    results.append({"item_name": product_name, "category_1_pred": c1, "category_2_pred": c2, "category_3_pred": c3})

pd.DataFrame(results).to_excel(OUT_FILE, index=False)
print(f"✅ 12B done → {OUT_FILE}")
del model, tokenizer; gc.collect()

In [ ]:

MODEL_ID = "mlx-community/gemma-4-26b-a4b-it-4bit"
OUT_FILE  = os.path.join(OUTPUT_DIR, "Gemma4_26B_MoE_Predictions.xlsx")

model, tokenizer = load_gemma4(MODEL_ID)
sanity_raw = generate(model, tokenizer,
    prompt=tokenizer.apply_chat_template([{"role": "user", "content": "Who are you?"}], add_generation_prompt=True),
    max_tokens=200, sampler=greedy_sampler, verbose=False)
print(f"Sanity stripped: {repr(strip_thinking(sanity_raw))}")

results = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="26B-MoE"):
    product_name = str(row.get(TITLE_COLUMN, "Unknown"))
    c1, c2, c3 = classify_product(product_name, taxonomy, model, tokenizer, greedy_sampler)
    results.append({"item_name": product_name, "category_1_pred": c1, "category_2_pred": c2, "category_3_pred": c3})

pd.DataFrame(results).to_excel(OUT_FILE, index=False)
print(f"✅ 26B MoE done → {OUT_FILE}")
del model, tokenizer; gc.collect()

In [3]:
MODEL_ID = "mlx-community/gemma-4-31b-it-4bit"
OUT_FILE  = os.path.join(OUTPUT_DIR, "Gemma4_31B_Predictions.xlsx")

model, tokenizer = load_gemma4(MODEL_ID)
sanity_raw = generate(model, tokenizer,
    prompt=tokenizer.apply_chat_template([{"role": "user", "content": "Who are you?"}], add_generation_prompt=True),
    max_tokens=200, sampler=greedy_sampler, verbose=False)
print(f"Sanity stripped: {repr(strip_thinking(sanity_raw))}")

results = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="31B"):
    product_name = str(row.get(TITLE_COLUMN, "Unknown"))
    c1, c2, c3 = classify_product(product_name, taxonomy, model, tokenizer, greedy_sampler)
    results.append({"item_name": product_name, "category_1_pred": c1, "category_2_pred": c2, "category_3_pred": c3})

pd.DataFrame(results).to_excel(OUT_FILE, index=False)
print(f"✅ 31B done → {OUT_FILE}")
del model, tokenizer; gc.collect()
print("\n🎉 ALL BENCHMARKS COMPLETE.")

  Loading mlx-community/gemma-4-31b-it-4bit...


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

[WARNING] Generating with a model that requires 16469 MB which is close to the maximum recommended size of 18186 MB. This can be slow. See the documentation for possible work-arounds: https://github.com/ml-explore/mlx-lm/tree/main#large-models
Sanity stripped: "I am a large language model, trained by Google. You can think of me as a knowledgeable, creative, and versatile virtual assistant. \n\nI don't have a physical body, feelings, or a personal history, but I can help you with a wide variety of tasks, such as:\n\n*   **Answering questions:** From simple facts to complex explanations.\n*   **Writing:** Creating emails, essays, stories, poems, or scripts.\n*   **Coding:** Writing and debugging code in many different languages.\n*   **"


31B:   0%|                                                                                      | 0/100 [00:00<?, ?it/s]

[WARNING] Generating with a model that requires 16469 MB which is close to the maximum recommended size of 18186 MB. This can be slow. See the documentation for possible work-arounds: https://github.com/ml-explore/mlx-lm/tree/main#large-models
[WARNING] Generating with a model that requires 16469 MB which is close to the maximum recommended size of 18186 MB. This can be slow. See the documentation for possible work-arounds: https://github.com/ml-explore/mlx-lm/tree/main#large-models
[WARNING] Generating with a model that requires 16469 MB which is close to the maximum recommended size of 18186 MB. This can be slow. See the documentation for possible work-arounds: https://github.com/ml-explore/mlx-lm/tree/main#large-models


...

[WARNING] Generating with a model that requires 16469 MB which is close to the maximum recommended size of 18186 MB. This can be slow. See the documentation for possible work-arounds: https://github.com/ml-explore/mlx-lm/tree/main#large-models
[WARNING] Generating with a model that requires 16469 MB which is close to the maximum recommended size of 18186 MB. This can be slow. See the documentation for possible work-arounds: https://github.com/ml-explore/mlx-lm/tree/main#large-models
[WARNING] Generating with a model that requires 16469 MB which is close to the maximum recommended size of 18186 MB. This can be slow. See the documentation for possible work-arounds: https://github.com/ml-explore/mlx-lm/tree/main#large-models


31B:  99%|████████████████████████████████████████████████████████████████████████████▏| 99/100 [42:18<00:30, 30.97s/it]

[WARNING] Generating with a model that requires 16469 MB which is close to the maximum recommended size of 18186 MB. This can be slow. See the documentation for possible work-arounds: https://github.com/ml-explore/mlx-lm/tree/main#large-models
[WARNING] Generating with a model that requires 16469 MB which is close to the maximum recommended size of 18186 MB. This can be slow. See the documentation for possible work-arounds: https://github.com/ml-explore/mlx-lm/tree/main#large-models
[WARNING] Generating with a model that requires 16469 MB which is close to the maximum recommended size of 18186 MB. This can be slow. See the documentation for possible work-arounds: https://github.com/ml-explore/mlx-lm/tree/main#large-models


31B: 100%|████████████████████████████████████████████████████████████████████████████| 100/100 [42:47<00:00, 25.67s/it]


✅ 31B done → /Users/nejam_abderrahmane/Downloads/Gemma4_Benchmarks/Gemma4_31B_Predictions.xlsx

🎉 ALL BENCHMARKS COMPLETE.
